# YemenVision-MTMC — المرحلة ٤: التقييم الفعلي على بيانات عامة من الإنترنت

هذا الدفتر يكمل دفتر المراحل ١–٣ ويجيب عن سؤال واحد: **هل النماذج المدرّبة تعمل فعلاً على بيانات لم ترها؟**

| المكوّن | مجموعة التقييم (تُنزَّل تلقائياً) | المقاييس |
|---|---|---|
| كاشف المركبات (YOLO) | COCO val2017 (تنزيل رسمي، مجموعة فرعية من المركبات) | mAP50 وmAP50-95 وPrecision وRecall + ثبات على ٣ أجزاء |
| ReID للمركبات | VeRi-776 (query/gallery) | mAP وCMC Rank-1/5/10 + فاصل ثقة 95% + مقارنة مع أوزان ImageNet |
| ربط المسارات بين الكاميرات | tracklets من VeRi-776 | IDF1 على مستوى المسار، Pairwise F1، ARI — مع ضبط العتبة على نصف الهويات وتقييم النصف الآخر |
| تتبع كاميرا واحدة (MOT) | اختياري: أي مجلد بصيغة MOTChallenge | MOTA وIDF1 وID switches |

## قواعد صارمة في هذا الدفتر
1. **لا أرقام تُكتب يدوياً.** كل رقم يخرج من تنفيذ فعلي ويُحفظ في JSON داخل Drive.
2. **لا ضبط على مجموعة التقييم.** العتبات تُختار على نصف منفصل من الهويات، ثم تُقاس على النصف الآخر.
3. **كل نتيجة تحمل حالة:** `PASS` أو `FAIL` أو `NOT_EVALUATED`. ما لم يُقَس لا يُدّعى.
4. **النماذج القديمة لا تُعدَّل أبداً** (قراءة فقط).

## حدود يجب أن تعرفها قبل التشغيل
- COCO وVeRi **ليستا بيانات يمنية**. نجاحك هنا يثبت أن النموذج سليم تقنياً ولا يثبت أنه يعمل على طرق اليمن. التأكيد النهائي يحتاج مجموعة تحقق يمنية صغيرة (≈200 صورة مُسمّاة) — الخلية الأخيرة تشرح كيف.
- نقطة البداية لكاشفك YOLO11s المُدرَّب مسبقاً على COCO، لذا اختبار COCO val يُعدّ أسهل من اختبار نطاق غريب تماماً. تعامل مع النتيجة كحد أعلى متفائل.
- ربط المسارات على VeRi هو **اختبار بديل (proxy)** لمرحلة الربط فقط: المسارات فيه نقية الهوية بالبناء، فلا يقيس أخطاء التتبع. MTMC الرسمي يحتاج CityFlowV2 (يتطلب موافقة اتفاقية).
- لا توجد مجموعة لوحات يمنية عامة، لذا **OCR اللوحات خارج هذه المرحلة** ويبقى على تسمياتك اليدوية.

In [1]:
# ١. الإعداد: Drive + المسارات + اكتشاف النماذج (قراءة فقط — لا استبدال)
from pathlib import Path
from datetime import datetime, timezone
import json, os, sys, re, hashlib, random, shutil, subprocess, time, importlib.util
import numpy as np

def status(kind, message):
    print({'ok': '✅', 'pause': '⏸️', 'error': '❌', 'warn': '⚠️'}.get(kind, 'ℹ️'), message)

try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
    DRIVE_ROOT = Path('/content/drive/MyDrive/YemenVision_MTMC')
    WORK = Path('/content/stage4_work')
except Exception:
    IN_COLAB = False
    DRIVE_ROOT = Path('./YemenVision_MTMC'); WORK = Path('./stage4_work')
    status('warn', 'لا يعمل داخل Colab؛ سيُستخدم مجلد محلي')

RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
EVAL_ROOT = DRIVE_ROOT / 'stage4_eval' / RUN_ID
for folder in (DRIVE_ROOT, WORK, EVAL_ROOT):
    folder.mkdir(parents=True, exist_ok=True)

def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, 'rb') as handle:
        for chunk in iter(lambda: handle.read(8 * 1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

def newest(paths):
    paths = [p for p in paths if p.is_file()]
    return max(paths, key=lambda p: p.stat().st_mtime) if paths else None

ARTIFACTS = DRIVE_ROOT / 'artifacts'
# --- اكتشاف النماذج: جذر Drive أولاً (غير تكراري) ثم artifacts. لا نبحث داخل runs حتى لا نلتقط أوزان تجربة عشوائية.
def find_in_root(patterns, extra_recursive=True):
    hits = []
    for pattern in patterns:
        hits += list(DRIVE_ROOT.glob(pattern))
    if not hits and extra_recursive and ARTIFACTS.exists():
        for pattern in patterns:
            hits += list(ARTIFACTS.rglob(pattern))
    return newest(hits)

# يمكنك تحديد المسار يدوياً هنا لتجاوز الاكتشاف التلقائي.
MANUAL_MODEL_PATHS = {'detector_baseline': '', 'detector_finetuned': '', 'reid': ''}

def pick(manual_key, patterns):
    manual = MANUAL_MODEL_PATHS.get(manual_key)
    if manual and Path(manual).is_file():
        return Path(manual)
    return find_in_root(patterns)

DETECTOR_MODELS = {}
baseline = pick('detector_baseline', ['best.pt', 'detection_yolo11s_best.pt'])
finetuned = pick('detector_finetuned', ['detector_finetuned_*_best.pt'])
if baseline: DETECTOR_MODELS['baseline'] = baseline
if finetuned: DETECTOR_MODELS['finetuned'] = finetuned
REID_MODEL = pick('reid', ['reid_bot_best.pt'])

PROVENANCE = {'run_id': RUN_ID, 'models': {}, 'datasets': {}}
for tag, path in list(DETECTOR_MODELS.items()) + [('reid', REID_MODEL)]:
    if path:
        PROVENANCE['models'][tag] = {'path': str(path), 'bytes': path.stat().st_size, 'sha256': sha256_file(path)}
        status('ok', f'{tag}: {path}')
    else:
        status('pause', f'{tag}: لم يُعثر على النموذج — سيُسجَّل NOT_EVALUATED')
status('ok', f'مخرجات هذه الجولة: {EVAL_ROOT}')

Mounted at /content/drive
✅ baseline: /content/drive/MyDrive/YemenVision_MTMC/detection_yolo11s_best.pt
✅ finetuned: /content/drive/MyDrive/YemenVision_MTMC/detector_finetuned_20261117T183727Z_best.pt
✅ reid: /content/drive/MyDrive/YemenVision_MTMC/reid_bot_best.pt
✅ مخرجات هذه الجولة: /content/drive/MyDrive/YemenVision_MTMC/stage4_eval/20261117T191946Z


In [2]:
# ٢. تثبيت ما ينقص فقط
required = {
    'ultralytics': 'ultralytics>=8.3.0', 'scipy': 'scipy', 'sklearn': 'scikit-learn',
    'requests': 'requests', 'tqdm': 'tqdm', 'PIL': 'Pillow', 'pandas': 'pandas',
    'motmetrics': 'motmetrics', 'kagglehub': 'kagglehub',
}
missing = [pkg for mod, pkg in required.items() if importlib.util.find_spec(mod) is None]
if missing:
    print('تثبيت:', missing)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q'] + missing, check=True)
import torch
status('ok', f'torch={torch.__version__} | CUDA={torch.cuda.is_available()}'
       + (f' | {torch.cuda.get_device_name(0)}' if torch.cuda.is_available() else ' — اختر Runtime > T4 GPU'))
DEVICE_ARG = 0 if torch.cuda.is_available() else 'cpu'

تثبيت: ['ultralytics>=8.3.0', 'motmetrics']
✅ torch=2.14.1+cu130 | CUDA=True | Tesla T4


In [3]:
# ٣. الإعدادات وبوابات القبول
# البوابات أهداف مشروعك القابلة للتعديل — ليست حقائق عالمية. اضبطها قبل التشغيل لا بعده.
CONFIG = {
    'seed': 42,
    'detector_imgsz': 640,
    'coco_subset_images': 1500,   # عدد صور COCO المسحوبة (كلها تحتوي مركبات)
    'coco_min_box_side_px': 10,   # تجاهل صناديق أصغر من هذا (صغيرة جداً لا تُقاس بإنصاف)
    'n_folds': 3,                 # لقياس ثبات النتيجة بين أجزاء منفصلة
    'reid_batch': 128,
    'reid_flip_tta': True,
    'compare_imagenet_baseline': True,
    'bootstrap_samples': 2000,
    'mtmc_thr_grid': [round(x, 3) for x in np.arange(0.10, 1.00, 0.025)],
    'mtmc_n_splits': 5,           # عدد التقسيمات العشوائية (dev/eval) للهويات
    'allow_kaggle_download': True,
}
GATES = {
    'detector_single_cls_map50_min': 0.50,   # «هل يجد المركبة؟» بغض النظر عن نوعها
    'detector_per_class_map50_min': 0.40,
    'detector_fold_std_max': 0.03,           # ثبات بين الأجزاء
    'reid_map_min': 0.50,
    'reid_rank1_min': 0.80,
    'reid_gain_over_imagenet_map_min': 0.15, # يثبت أن التدريب أضاف قيمة فعلية
    'mtmc_proxy_idf1_min': 0.60,
    'mot_idf1_min': 0.60,
}
random.seed(CONFIG['seed']); np.random.seed(CONFIG['seed']); torch.manual_seed(CONFIG['seed'])
(EVAL_ROOT / 'config.json').write_text(json.dumps({'CONFIG': CONFIG, 'GATES': GATES}, indent=2, default=str), encoding='utf-8')
status('ok', 'الإعدادات والبوابات محفوظة')

✅ الإعدادات والبوابات محفوظة


In [4]:
# ٤. أدوات تنزيل موثوقة: إعادة محاولة + فحص الحجم + SHA256 مسجَّل + فك آمن (بلا path traversal)
import requests, zipfile
from concurrent.futures import ThreadPoolExecutor

def download_file(url, dest, retries=4, expected_sha256=None, timeout=60):
    dest = Path(dest); dest.parent.mkdir(parents=True, exist_ok=True)
    if dest.is_file() and dest.stat().st_size > 0:
        if not expected_sha256 or sha256_file(dest) == expected_sha256.lower():
            return dest
        dest.unlink()
    last = None
    for attempt in range(1, retries + 1):
        tmp = dest.with_suffix(dest.suffix + '.part')
        try:
            with requests.get(url, stream=True, timeout=timeout) as response:
                response.raise_for_status()
                total = int(response.headers.get('Content-Length', 0))
                with open(tmp, 'wb') as handle:
                    for chunk in response.iter_content(1 << 20):
                        if chunk:
                            handle.write(chunk)
            if total and tmp.stat().st_size != total:
                raise IOError(f'حجم ناقص {tmp.stat().st_size}/{total}')
            if expected_sha256 and sha256_file(tmp) != expected_sha256.lower():
                raise IOError('SHA256 غير مطابق')
            tmp.replace(dest)
            return dest
        except Exception as exc:
            last = exc
            tmp.unlink(missing_ok=True)
            time.sleep(2 * attempt)
    raise RuntimeError(f'فشل تنزيل {url}: {last}')

def extract_one_member(zip_path, member_suffix, out_dir):
    # يستخرج ملفاً واحداً فقط ويكتبه باسمه المجرد (لا مسارات من داخل الأرشيف)
    out_dir = Path(out_dir); out_dir.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path) as archive:
        for info in archive.infolist():
            if info.filename.endswith(member_suffix) and not info.is_dir():
                target = out_dir / Path(info.filename).name
                with archive.open(info) as src, open(target, 'wb') as dst:
                    shutil.copyfileobj(src, dst)
                return target
    raise FileNotFoundError(member_suffix)

status('ok', 'أدوات التنزيل جاهزة')

✅ أدوات التنزيل جاهزة


## دوال المقاييس
هذه الخلية لا تعتمد على أي إنترنت أو GPU، وقد اختُبرت على بيانات مصطنعة ذات إجابة معروفة قبل تسليم الدفتر.

In [5]:
# ٥. دوال المقاييس (numpy/scipy/sklearn فقط)
import numpy as np
from scipy.optimize import linear_sum_assignment

def bootstrap_mean_ci(values, n=2000, seed=0, alpha=0.05):
    values = np.asarray(values, dtype=np.float64)
    if len(values) == 0:
        return {'mean': None, 'ci_low': None, 'ci_high': None, 'n': 0}
    rng = np.random.default_rng(seed)
    means = values[rng.integers(0, len(values), size=(n, len(values)))].mean(axis=1)
    return {'mean': float(values.mean()), 'ci_low': float(np.quantile(means, alpha / 2)),
            'ci_high': float(np.quantile(means, 1 - alpha / 2)), 'n': int(len(values))}

def evaluate_reid(qf, gf, q_pid, g_pid, q_cam, g_cam, topk=(1, 5, 10)):
    # بروتوكول VeRi/Market القياسي: تُحذف من gallery الصور ذات نفس الهوية ونفس الكاميرا (وكذلك pid<0)
    qf, gf = np.asarray(qf, np.float32), np.asarray(gf, np.float32)
    q_pid, g_pid, q_cam, g_cam = map(np.asarray, (q_pid, g_pid, q_cam, g_cam))
    sim = qf @ gf.T
    K = max(topk)
    ap_list, cmc_rows, kept_idx, first_wrong = [], [], [], {}
    for i in range(len(qf)):
        order = np.argsort(-sim[i], kind='stable')
        junk = ((g_pid[order] == q_pid[i]) & (g_cam[order] == q_cam[i])) | (g_pid[order] < 0)
        keep = ~junk
        ranked = order[keep]
        matches = (g_pid[ranked] == q_pid[i]).astype(np.int64)
        if matches.sum() == 0:
            continue                                    # لا توجد صورة صحيحة من كاميرا أخرى: يُتجاهل الاستعلام
        hits = np.cumsum(matches)
        precision = hits / (np.arange(len(matches)) + 1)
        ap_list.append(float((precision * matches).sum() / matches.sum()))
        cmc = np.minimum(hits, 1)[:K]
        if len(cmc) < K:
            cmc = np.concatenate([cmc, np.ones(K - len(cmc), dtype=cmc.dtype)])
        cmc_rows.append(cmc)
        kept_idx.append(i)
        if matches[0] == 0:
            first_wrong[i] = int(ranked[0])
    if not ap_list:
        raise ValueError('لا يوجد أي استعلام صالح (تحقق من ملفات query/gallery)')
    cmc_arr = np.stack(cmc_rows)
    out = {'mAP': float(np.mean(ap_list)), 'n_valid_queries': len(ap_list), 'n_queries': int(len(qf)),
           'per_query_ap': ap_list, 'kept_query_index': kept_idx, 'first_wrong_gallery_index': first_wrong}
    for k in topk:
        out[f'rank{k}'] = float(cmc_arr[:, k - 1].mean())
    out['per_query_rank1'] = cmc_arr[:, 0].astype(float).tolist()
    return out

def _pairs(n):
    return n * (n - 1) // 2

def cluster_metrics(gt, pred, weights=None):
    # gt/pred: تسميات لكل tracklet. weights: عدد الصور/الإطارات في كل tracklet.
    from sklearn.metrics import adjusted_rand_score
    gt, pred = np.asarray(gt), np.asarray(pred)
    weights = np.ones(len(gt)) if weights is None else np.asarray(weights, dtype=np.float64)
    g_ids, g_inv = np.unique(gt, return_inverse=True)
    p_ids, p_inv = np.unique(pred, return_inverse=True)
    contingency = np.zeros((len(g_ids), len(p_ids)))
    wmat = np.zeros((len(g_ids), len(p_ids)))
    for a, b, w in zip(g_inv, p_inv, weights):
        contingency[a, b] += 1
        wmat[a, b] += w
    tp = sum(_pairs(int(v)) for v in contingency.ravel())
    pred_pairs = sum(_pairs(int(v)) for v in contingency.sum(axis=0))
    gt_pairs = sum(_pairs(int(v)) for v in contingency.sum(axis=1))
    precision = tp / pred_pairs if pred_pairs else 1.0
    recall = tp / gt_pairs if gt_pairs else 1.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0
    # IDF1 على مستوى tracklet: مطابقة مجرية بين هويات الحقيقة والعناقيد بوزن الصور.
    # مجموع الأوزان متساوٍ للجهتين، لذا IDF1 = IDTP / الإجمالي.
    rows, cols = linear_sum_assignment(-wmat)
    idtp = wmat[rows, cols].sum()
    idf1 = float(idtp / weights.sum()) if weights.sum() else 0.0
    return {'pairwise_precision': float(precision), 'pairwise_recall': float(recall), 'pairwise_f1': float(f1),
            'tracklet_idf1': idf1, 'ari': float(adjusted_rand_score(gt, pred)),
            'n_gt_ids': int(len(g_ids)), 'n_pred_clusters': int(len(p_ids)), 'n_tracklets': int(len(gt))}

def cluster_tracklets(embeddings, cams, t_start, t_end, threshold, camera_aware=True):
    # تجميع هرمي (average linkage) على المسافة الكوسينية. camera_aware: يمنع دمج مسارين متداخلين زمنياً في نفس الكاميرا.
    from scipy.cluster.hierarchy import linkage, fcluster
    from scipy.spatial.distance import squareform
    emb = np.asarray(embeddings, np.float32)
    n = len(emb)
    if n == 1:
        return np.array([1])
    dist = np.clip(1.0 - emb @ emb.T, 0.0, 2.0).astype(np.float64)
    if camera_aware:
        cams, t_start, t_end = np.asarray(cams), np.asarray(t_start), np.asarray(t_end)
        same_cam = cams[:, None] == cams[None, :]
        overlap = (t_start[:, None] <= t_end[None, :]) & (t_start[None, :] <= t_end[:, None])
        dist[same_cam & overlap] = 2.0
    np.fill_diagonal(dist, 0.0)
    dist = (dist + dist.T) / 2
    tree = linkage(squareform(dist, checks=False), method='average')
    return fcluster(tree, t=threshold, criterion='distance')

status('ok', 'دوال المقاييس جاهزة')

✅ دوال المقاييس جاهزة


## أ) كاشف المركبات — COCO val2017 (تنزيل رسمي)
- نُنزّل ملف التسميات الرسمي، ثم **صور العيّنة فقط** (لا نُنزّل 778MB كاملة).
- نربط فئات COCO (car/bus/truck/motorcycle/bicycle) بفئات كاشفك حسب الاسم، ونطبع الربط ليُراجَع.
- نختار صوراً لا تحتوي صناديق `iscrowd` (غامضة)، ونُسقط الصناديق الدقيقة جداً.
- نقيس مرتين: **حسب الفئة** و**بلا فئة (single_cls)** — الثانية تجيب «هل يجد المركبة؟» دون أن يتأثر الرقم باختلاف تعريف الفئات بين COCO ومجموعتك.
- نكرر القياس على ٣ أجزاء منفصلة لإظهار الثبات.

In [6]:
# ٦. بناء مجموعة تقييم المركبات من COCO val2017
import json, random
from collections import Counter

COCO_ANN_URL = 'http://images.cocodataset.org/annotations/annotations_trainval2017.zip'
VEHICLE_COCO = ['bicycle', 'car', 'motorcycle', 'bus', 'truck']
SYNONYMS = {
    'bicycle': ['bicycle', 'cycle'],
    'car': ['car', 'sedan', 'suv', 'van', 'taxi', 'pickup', 'jeep', 'hatchback', 'vehicle'],
    'motorcycle': ['motorcycle', 'motorbike', 'moped', 'scooter'],
    'bus': ['bus', 'minibus'],
    'truck': ['truck', 'lorry'],
}
MANUAL_CLASS_MAP = {}   # مثال: {'car': 2, 'bus': 5} بأرقام فئات كاشفك — يتجاوز الربط التلقائي

def map_coco_to_detector(det_names, manual=None):
    low = [str(n).lower().strip() for n in det_names]
    if len(low) == 1:                                   # كاشف بفئة واحدة: كل المركبات = الفئة 0
        return {c: 0 for c in VEHICLE_COCO}, {}
    mapping, ambiguous = {}, {}
    for coco_name in VEHICLE_COCO:
        if manual and coco_name in manual:
            mapping[coco_name] = int(manual[coco_name]); continue
        candidates = [i for i, n in enumerate(low) if n == coco_name or n in SYNONYMS[coco_name]]
        if candidates:
            exact = [i for i in candidates if low[i] == coco_name]
            mapping[coco_name] = (exact or candidates)[0]
            if len(candidates) > 1:
                ambiguous[coco_name] = [det_names[i] for i in candidates]
    return mapping, ambiguous

def coco_bbox_to_yolo(bbox, width, height, min_side=0):
    x, y, w, h = bbox
    x1, y1 = max(0.0, x), max(0.0, y)
    x2, y2 = min(float(width), x + w), min(float(height), y + h)
    bw, bh = x2 - x1, y2 - y1
    if bw < min_side or bh < min_side or bw <= 0 or bh <= 0:
        return None
    return ((x1 + bw / 2) / width, (y1 + bh / 2) / height, bw / width, bh / height)

# ---- تنزيل التسميات (≈241MB مرة واحدة) واستخراج ملف val فقط
ann_zip = download_file(COCO_ANN_URL, WORK / 'coco' / 'annotations_trainval2017.zip')
ann_json = extract_one_member(ann_zip, 'instances_val2017.json', WORK / 'coco')
coco = json.loads(ann_json.read_text(encoding='utf-8'))
PROVENANCE['datasets']['coco_val2017_annotations'] = {
    'url': COCO_ANN_URL, 'sha256': sha256_file(ann_zip),
    'license_note': 'تسميات COCO بترخيص CC BY 4.0؛ الصور من Flickr بشروطها الخاصة — للتقييم البحثي فقط'}
cat_name = {c['id']: c['name'] for c in coco['categories']}
images_by_id = {im['id']: im for im in coco['images']}
anns_by_image = {}
for ann in coco['annotations']:
    anns_by_image.setdefault(ann['image_id'], []).append(ann)
status('ok', f"COCO val2017: {len(images_by_id)} صورة | {len(coco['annotations'])} تسمية")

✅ COCO val2017: 5000 صورة | 36781 تسمية


In [7]:
# ٧. ربط الفئات + سحب العيّنة + تنزيل الصور + كتابة labels بصيغة YOLO
from ultralytics import YOLO

if not DETECTOR_MODELS:
    status('pause', 'لا يوجد كاشف — يُتخطى قسم الكاشف')
    COCO_EVAL = None
else:
    probe_path = next(iter(DETECTOR_MODELS.values()))
    probe = YOLO(str(probe_path))
    det_names = [probe.names[i] for i in sorted(probe.names)]
    class_map, ambiguous = map_coco_to_detector(det_names, MANUAL_CLASS_MAP)
    print('فئات كاشفك:', det_names)
    print('الربط COCO → كاشفك:', {k: f'{v}:{det_names[v]}' for k, v in class_map.items()})
    unmapped = [c for c in VEHICLE_COCO if c not in class_map]
    if unmapped: status('warn', f'فئات COCO بلا مقابل في كاشفك (تُسقط من الحقيقة): {unmapped}')
    if ambiguous: status('warn', f'التباس (أكثر من فئة لكاشفك تطابق فئة COCO): {ambiguous} — استخدم MANUAL_CLASS_MAP')
    for tag, p in DETECTOR_MODELS.items():              # الكاشفان يجب أن يشتركا في ترتيب الفئات
        names_i = YOLO(str(p)).names
        if [names_i[i] for i in sorted(names_i)] != det_names:
            status('error', f'ترتيب فئات {tag} يختلف عن baseline — قارن كلاً على حدة')

    vehicle_ids = {cid for cid, n in cat_name.items() if n in VEHICLE_COCO}
    eligible = []
    for image_id, anns in anns_by_image.items():
        veh = [a for a in anns if a['category_id'] in vehicle_ids]
        if not veh or any(a.get('iscrowd', 0) for a in veh):
            continue
        im = images_by_id[image_id]
        usable = [a for a in veh if cat_name[a['category_id']] in class_map
                  and coco_bbox_to_yolo(a['bbox'], im['width'], im['height'], CONFIG['coco_min_box_side_px'])]
        if usable:
            eligible.append(image_id)
    eligible.sort()
    chosen = sorted(random.Random(CONFIG['seed']).sample(eligible, min(CONFIG['coco_subset_images'], len(eligible))))
    status('ok', f'صور مؤهلة: {len(eligible)} | المختارة: {len(chosen)}')

    EVAL_DIR = WORK / 'coco_vehicle_eval'
    (EVAL_DIR / 'images' / 'val').mkdir(parents=True, exist_ok=True)
    (EVAL_DIR / 'labels' / 'val').mkdir(parents=True, exist_ok=True)

    def fetch(image_id):
        im = images_by_id[image_id]
        target = EVAL_DIR / 'images' / 'val' / im['file_name']
        try:
            download_file(im['coco_url'], target, retries=3)
            return image_id, True
        except Exception:
            return image_id, False

    with ThreadPoolExecutor(8) as pool:
        results = list(pool.map(fetch, chosen))
    ok_ids = [i for i, ok in results if ok]
    status('ok' if len(ok_ids) == len(chosen) else 'warn', f'نُزّلت {len(ok_ids)}/{len(chosen)} صورة')

    instance_counts, final_paths = Counter(), []
    for image_id in ok_ids:
        im = images_by_id[image_id]
        lines = []
        for a in anns_by_image[image_id]:
            name = cat_name[a['category_id']]
            if name not in class_map:
                continue
            box = coco_bbox_to_yolo(a['bbox'], im['width'], im['height'], CONFIG['coco_min_box_side_px'])
            if box:
                lines.append(f"{class_map[name]} {box[0]:.6f} {box[1]:.6f} {box[2]:.6f} {box[3]:.6f}")
                instance_counts[name] += 1
        if lines:
            stem = Path(im['file_name']).stem
            (EVAL_DIR / 'labels' / 'val' / f'{stem}.txt').write_text('\n'.join(lines) + '\n', encoding='utf-8')
            final_paths.append(str((EVAL_DIR / 'images' / 'val' / im['file_name']).resolve()))

    import yaml
    def write_split(name, paths):
        (EVAL_DIR / f'{name}.txt').write_text('\n'.join(paths) + '\n', encoding='utf-8')
        cfg = {'path': str(EVAL_DIR.resolve()), 'train': f'{name}.txt', 'val': f'{name}.txt',
               'names': {i: n for i, n in enumerate(det_names)}}
        (EVAL_DIR / f'{name}.yaml').write_text(yaml.safe_dump(cfg, allow_unicode=True), encoding='utf-8')
        return EVAL_DIR / f'{name}.yaml'

    COCO_EVAL = {'all': write_split('all', final_paths)}
    for k in range(CONFIG['n_folds']):
        COCO_EVAL[f'fold{k}'] = write_split(f'fold{k}', final_paths[k::CONFIG['n_folds']])
    COCO_EVAL_INFO = {'n_images': len(final_paths), 'instances_by_coco_class': dict(instance_counts),
                      'class_map': {k: int(v) for k, v in class_map.items()}, 'unmapped_coco_classes': unmapped,
                      'ambiguous': ambiguous, 'seed': CONFIG['seed'],
                      'min_box_side_px': CONFIG['coco_min_box_side_px']}
    print(json.dumps(COCO_EVAL_INFO, ensure_ascii=False, indent=2))

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings
فئات كاشفك: ['ambulance', 'army vehicle', 'auto rickshaw', 'bicycle', 'bus', 'car', 'garbagevan', 'human hauler', 'minibus', 'minivan', 'motorbike', 'pickup', 'policecar', 'rickshaw', 'scooter', 'suv', 'taxi', 'three wheelers -CNG-', 'truck', 'van', 'wheelbarrow']
الربط COCO → كاشفك: {'bicycle': '3:bicycle', 'car': '5:car', 'motorcycle': '10:motorbike', 'bus': '4:bus', 'truck': '18:truck'}
⚠️ التباس (أكثر من فئة لكاشفك تطابق فئة COCO): {'car': ['car', 'pickup', 'suv', 'taxi', 'van'], 'motorcycle': ['motorbike', 'scooter'], 'bus': ['bus', 'minibus']} — استخدم MANUAL_CLASS_MAP
✅ صور مؤهلة: 831 | المختارة: 831
✅ نُزّلت 831/831 صورة
{
  "n_images": 831,
  "instances_by_coco_class": {
    "truck": 379,
 

In [8]:
# ٨. تشغيل التقييم: كل كاشف × (الكل بالفئات، بلا فئة، ٣ أجزاء)
def run_det_val(model_path, yaml_path, tag, single_cls=False):
    model = YOLO(str(model_path))
    res = model.val(data=str(yaml_path), imgsz=CONFIG['detector_imgsz'], batch=16, conf=0.001, iou=0.6,
                    device=DEVICE_ARG, plots=False, verbose=False, single_cls=single_cls,
                    project=str(EVAL_ROOT / 'ultralytics'), name=tag, exist_ok=True)
    box = res.box
    names = model.names
    per_class = {}
    try:
        for idx, ap in zip(box.ap_class_index, box.ap50):
            per_class[str(names[int(idx)])] = float(ap)
    except Exception:
        pass
    return {'precision': float(box.mp), 'recall': float(box.mr), 'map50': float(box.map50),
            'map50_95': float(box.map), 'per_class_ap50': per_class, 'speed_ms': dict(res.speed)}

DETECTOR_RESULTS = {}
if COCO_EVAL:
    for tag, path in DETECTOR_MODELS.items():
        entry = {'checkpoint': str(path), 'sha256': PROVENANCE['models'][tag]['sha256']}
        entry['per_class_all'] = run_det_val(path, COCO_EVAL['all'], f'{tag}_all')
        entry['single_cls_all'] = run_det_val(path, COCO_EVAL['all'], f'{tag}_single', single_cls=True)
        folds = [run_det_val(path, COCO_EVAL[f'fold{k}'], f'{tag}_fold{k}') for k in range(CONFIG['n_folds'])]
        values = np.array([f['map50'] for f in folds])
        entry['folds_map50'] = values.tolist()
        entry['folds_map50_mean'] = float(values.mean())
        entry['folds_map50_std'] = float(values.std(ddof=1)) if len(values) > 1 else 0.0
        DETECTOR_RESULTS[tag] = entry
        a, s = entry['per_class_all'], entry['single_cls_all']
        print(f"\n[{tag}] حسب الفئة: P={a['precision']:.3f} R={a['recall']:.3f} mAP50={a['map50']:.3f} mAP50-95={a['map50_95']:.3f}")
        print(f"[{tag}] بلا فئة : P={s['precision']:.3f} R={s['recall']:.3f} mAP50={s['map50']:.3f}")
        print(f"[{tag}] الأجزاء: {np.round(values, 3).tolist()} ± {entry['folds_map50_std']:.3f}")
    (EVAL_ROOT / 'detector_eval.json').write_text(
        json.dumps({'dataset': COCO_EVAL_INFO, 'results': DETECTOR_RESULTS}, ensure_ascii=False, indent=2), encoding='utf-8')
    status('ok', 'حُفظ detector_eval.json')
else:
    status('pause', 'قسم الكاشف NOT_EVALUATED')

Ultralytics 8.4.215 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (Tesla T4, 14913MiB)
YOLO11s summary (fused): 100 layers, 9,420,927 parameters, 0 gradients, 21.5 GFLOPs
val: Fast image access ✅ (ping: 0.1±0.2 ms, read: 1690.1±495.2 MB/s, size: 112.8 KB)
val: Scanning /content/stage4_work/coco_vehicle_eval/labels/val... 831 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 831/831 1.2Kit/s 0.7s
val: New cache created: /content/stage4_work/coco_vehicle_eval/labels/val.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 52/52 3.2it/s 16.3s
                   all        831       2636      0.479      0.315      0.314      0.185
Speed: 2.0ms preprocess, 6.8ms inference, 0.0ms loss, 3.9ms postprocess per image
Ultralytics 8.4.215 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (Tesla T4, 14913MiB)
YOLO11s summary (fused): 100 layers, 9,420,927 parameters, 0 gradients, 21.5 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1700.

## ب) ReID — VeRi-776 (بروتوكول query/gallery القياسي)
- يبحث الدفتر أولاً عن VeRi في Drive (`ReID_uploads`)، وإلا يحاول Kaggle كما في دفترك الأول (يحتاج `KAGGLE_USERNAME` و`KAGGLE_KEY` في Colab Secrets).
- يُفحص **تسرب البيانات**: لا يجوز أن تشترك هويات التدريب مع query/gallery.
- المقارنة مع **أوزان ImageNet بلا تدريب ReID** تثبت أن تدريبك أضاف قيمة فعلية، لا أن الشبكة قوية أصلاً.

In [14]:
# ٩. العثور على VeRi وفهرستها وفحص التسرب
import pandas as pd
VERI_LOCAL_DIR = ''   # مثال: '/content/drive/MyDrive/YemenVision_MTMC/ReID_uploads/VeRi' — يتجاوز البحث
VERI_KAGGLE_SLUGS = ['abhyudaya12/veri-vehicle-re-identification-dataset']
VERI_RE = re.compile(r'(\d{4})_c(\d{3})_(\d+)_(\d+)\.(?:jpg|jpeg|png)', re.I)
IMG_EXT = {'.jpg', '.jpeg', '.png'}

def parse_veri_name(name):
    m = VERI_RE.search(str(name))
    return (int(m.group(1)), int(m.group(2)), int(m.group(3))) if m else None

def veri_dirs_in(root):
    for current, dirs, _ in os.walk(root):
        depth = Path(current).relative_to(root).parts
        if len(depth) > 5:
            dirs[:] = []; continue
        if 'image_query' in dirs and 'image_test' in dirs:
            return Path(current)
    return None

def setup_kaggle_credentials():
    # تفعيل بيئة Kaggle عبر فحص الملف المرفوع أو Secrets لتفادي الخطأ
    uploaded_json = Path('/content/kaggle.json')
    kaggle_dir = Path.home() / '.kaggle'
    if uploaded_json.is_file():
        kaggle_dir.mkdir(parents=True, exist_ok=True)
        shutil.copy(uploaded_json, kaggle_dir / 'kaggle.json')
        (kaggle_dir / 'kaggle.json').chmod(0o600)
        status('ok', 'تم اكتشاف ملف kaggle.json المرفوع وتهيئته بنجاح.')

    if IN_COLAB:
        from google.colab import userdata
        try:
            os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
            os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')
        except Exception:
            if not (kaggle_dir / 'kaggle.json').is_file():
                status('warn', 'لم يتم العثور على مفاتيح Kaggle في Colab Secrets أو كملف مرفوع. يرجى رفع ملف kaggle.json لإتمام التنزيل.')

def find_veri():
    roots = [Path(VERI_LOCAL_DIR)] if VERI_LOCAL_DIR else [
        DRIVE_ROOT / 'ReID_uploads', DRIVE_ROOT / 'datasets', WORK, Path('/content/kagglehub_cache'),
        Path.home() / '.cache' / 'kagglehub']
    for root in roots:
        if root.exists():
            hit = veri_dirs_in(root)
            if hit: return hit
    if CONFIG['allow_kaggle_download']:
        try:
            setup_kaggle_credentials()
            import kagglehub
            for slug in VERI_KAGGLE_SLUGS:
                downloaded = Path(kagglehub.dataset_download(slug))
                hit = veri_dirs_in(downloaded)
                if hit:
                    PROVENANCE['datasets']['veri776'] = {'source': f'kaggle:{slug}',
                        'license_note': 'VeRi-776 للأغراض البحثية؛ تحقق من شروط المصدر الأصلي ومن أن نسخة Kaggle مطابقة'}
                    return hit
        except Exception as exc:
            status('warn', f'تعذر تنزيل VeRi تلقائياً: {type(exc).__name__}: {exc}')
    return None

VERI_ROOT = find_veri()
VERI = None
if VERI_ROOT is None:
    status('pause', 'لم تُعثر VeRi. ضعها في Drive/ReID_uploads أو ارفع ملف kaggle.json الخاص بك أو أضف مفاتيح Kaggle في Secrets ثم أعد التشغيل — ReID وMTMC NOT_EVALUATED')
else:
    status('ok', f'VeRi: {VERI_ROOT}')
    PROVENANCE['datasets'].setdefault('veri776', {'source': str(VERI_ROOT)})
    rows = []
    for split, folder in (('train', 'image_train'), ('query', 'image_query'), ('gallery', 'image_test')):
        d = VERI_ROOT / folder
        if not d.is_dir(): continue
        for p in sorted(d.iterdir()):
            if p.suffix.lower() in IMG_EXT and parse_veri_name(p.name):
                pid, cam, ts = parse_veri_name(p.name)
                rows.append({'path': str(p), 'name': p.name, 'pid': pid, 'cam': cam, 'ts': ts, 'split': split})
    VERI = pd.DataFrame(rows)
    print(VERI.groupby('split').agg(images=('path', 'count'), ids=('pid', 'nunique'), cams=('cam', 'nunique')))
    tr_ids = set(VERI[VERI.split == 'train'].pid); te_ids = set(VERI[VERI.split != 'train'].pid)
    overlap_ids = tr_ids & te_ids
    overlap_names = set(VERI[VERI.split == 'train'].name) & set(VERI[VERI.split != 'train'].name)
    LEAKAGE = {'train_test_identity_overlap': len(overlap_ids), 'train_test_filename_overlap': len(overlap_names)}
    q_ids = set(VERI[VERI.split == 'query'].pid); g_ids = set(VERI[VERI.split == 'gallery'].pid)
    LEAKAGE['query_ids_missing_from_gallery'] = len(q_ids - g_ids)
    print('فحص التسرب:', LEAKAGE)
    if len(tr_ids) and overlap_ids:
        status('error', 'هويات التدريب تتقاطع مع الاختبار — النتائج غير صالحة كإثبات تعميم')
    elif not len(tr_ids):
        status('warn', 'لا يوجد image_train: لا يمكن فحص التسرب مع التدريب')
    else:
        status('ok', 'لا تقاطع بين هويات التدريب والاختبار')

⚠️ لم يتم العثور على مفاتيح Kaggle في Colab Secrets أو كملف مرفوع. يرجى رفع ملف kaggle.json لإتمام التنزيل.


100%|██████████| 946M/946M [00:11<00:00, 86.9MB/s]

Extracting files...


✅ VeRi: /root/.cache/kagglehub/datasets/abhyudaya12/veri-vehicle-re-identification-dataset/versions/1/VeRi
         images  ids  cams
split                     
gallery   11579  200    19
query      1678  200    19
train     37778  576    20
فحص التسرب: {'train_test_identity_overlap': 0, 'train_test_filename_overlap': 0, 'query_ids_missing_from_gallery': 0}
✅ لا تقاطع بين هويات التدريب والاختبار


In [17]:
# ١٠. تقييم ReID: نموذجك مقابل أوزان ImageNet — mAP وCMC وفاصل ثقة وأخطاء
import torch
import torch.nn as nn, torch.nn.functional as F
from torchvision import models, transforms
from torch.utils.data import Dataset, DataLoader
from PIL import Image

# تعريف DEVICE لتجنب NameError
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

class LegacyReIDNet(nn.Module):
    # مطابق لبنية الدفتر الأول: ResNet backbone + pool + BNNeck. وقت الاستدلال يُرجع bn(features) كما في التدريب.
    def __init__(self, backbone, n_classes=2, use_bn=True, pretrained=False):
        super().__init__()
        weights = None
        if pretrained:
            weights = getattr(models, {'resnet18': 'ResNet18_Weights', 'resnet34': 'ResNet34_Weights',
                                       'resnet50': 'ResNet50_Weights'}[backbone]).DEFAULT
        base = getattr(models, backbone)(weights=weights)
        self.dim = base.fc.in_features
        self.backbone = nn.Sequential(*list(base.children())[:-2])
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.use_bn = use_bn
        self.bn = nn.BatchNorm1d(self.dim)
        self.classifier = nn.Linear(self.dim, n_classes, bias=False)
    def forward(self, x):
        f = self.pool(self.backbone(x)).flatten(1)
        return self.bn(f) if self.use_bn else f

class ImgDataset(Dataset):
    def __init__(self, paths, size_hw):
        self.paths = list(paths)
        self.tf = transforms.Compose([transforms.Resize(tuple(size_hw)), transforms.ToTensor(),
                                      transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])])
    def __len__(self): return len(self.paths)
    def __getitem__(self, i):
        with Image.open(self.paths[i]) as im:
            return self.tf(im.convert('RGB'))

@torch.no_grad()
def extract_embeddings(net, paths, size_hw, batch=128, flip=True):
    net = net.to(DEVICE).eval()
    loader = DataLoader(ImgDataset(paths, size_hw), batch_size=batch, shuffle=False, num_workers=2, pin_memory=True)
    outs = []
    for x in loader:
        x = x.to(DEVICE, non_blocking=True)
        with torch.autocast('cuda', dtype=torch.float16, enabled=DEVICE.type == 'cuda'):
            f = net(x).float()
            if flip:
                f = f + net(torch.flip(x, dims=[3])).float()
        outs.append(F.normalize(f, dim=1).cpu())
    return torch.cat(outs).numpy()

REID_RESULTS, EMB = {}, {}
if VERI is None or REID_MODEL is None:
    status('pause', 'ReID NOT_EVALUATED (يلزم VeRi ونموذج reid_bot_best.pt)')
else:
    ck = torch.load(REID_MODEL, map_location='cpu', weights_only=False)
    cfg = ck.get('cfg', {})
    backbone, size_hw = cfg.get('backbone', 'resnet50'), tuple(cfg.get('size', (224, 224)))
    net = LegacyReIDNet(backbone, int(cfg.get('n_cls', 2)))
    incompat = net.load_state_dict(ck['state_dict'], strict=False)
    bad = [k for k in incompat.missing_keys if k.startswith(('backbone.', 'bn.'))]
    if bad:
        raise RuntimeError(f'أوزان ناقصة في checkpoint ({len(bad)}): {bad[:3]} — لن نقيس نموذجاً محمّلاً جزئياً')
    status('ok', f'حُمّل {REID_MODEL.name} | backbone={backbone} | size={size_hw} | unexpected={len(incompat.unexpected_keys)}')

    q = VERI[VERI.split == 'query'].reset_index(drop=True)
    g = VERI[VERI.split == 'gallery'].reset_index(drop=True)
    def run_one(tag, model):
        qf = extract_embeddings(model, q.path, size_hw, CONFIG['reid_batch'], CONFIG['reid_flip_tta'])
        gf = extract_embeddings(model, g.path, size_hw, CONFIG['reid_batch'], CONFIG['reid_flip_tta'])
        res = evaluate_reid(qf, gf, q.pid.values, g.pid.values, q.cam.values, g.cam.values)
        ci_map = bootstrap_mean_ci(res['per_query_ap'], CONFIG['bootstrap_samples'], CONFIG['seed'])
        ci_r1 = bootstrap_mean_ci(res['per_query_rank1'], CONFIG['bootstrap_samples'], CONFIG['seed'])
        kept = res['kept_query_index']
        by_cam = {}
        for cam in sorted(q.cam.unique()):
            sel = [j for j, qi in enumerate(kept) if q.cam.values[qi] == cam]
            if sel:
                by_cam[int(cam)] = {'mAP': float(np.mean([res['per_query_ap'][j] for j in sel])),
                                    'rank1': float(np.mean([res['per_query_rank1'][j] for j in sel])), 'n': len(sel)}
        failures = []
        for qi, gi in list(res['first_wrong_gallery_index'].items())[:50]:
            failures.append({'query': q.name[qi], 'wrong_top1': g.name[gi], 'query_id': int(q.pid[qi]), 'wrong_id': int(g.pid[gi])})
        summary = {'mAP': res['mAP'], 'mAP_ci95': [ci_map['ci_low'], ci_map['ci_high']],
                   'rank1': res['rank1'], 'rank1_ci95': [ci_r1['ci_low'], ci_r1['ci_high']],
                   'rank5': res['rank5'], 'rank10': res['rank10'],
                   'n_valid_queries': res['n_valid_queries'], 'n_gallery': int(len(g)),
                   'by_query_camera': by_cam, 'sample_failures': failures}
        print(f"[{tag}] mAP={summary['mAP']:.4f} (95% CI {summary['mAP_ci95'][0]:.4f}–{summary['mAP_ci95'][1]:.4f}) | "
              f"R1={summary['rank1']:.4f} R5={summary['rank5']:.4f} R10={summary['rank10']:.4f} | queries={summary['n_valid_queries']}")
        return summary, (qf, gf)

    REID_RESULTS['trained'], EMB['trained'] = run_one('نموذجك', net)
    REID_RESULTS['trained']['checkpoint'] = str(REID_MODEL)
    REID_RESULTS['trained']['sha256'] = PROVENANCE['models']['reid']['sha256']
    REID_RESULTS['trained']['leakage_check'] = LEAKAGE
    if CONFIG['compare_imagenet_baseline']:
        try:
            base_net = LegacyReIDNet(backbone, use_bn=False, pretrained=True)
            REID_RESULTS['imagenet_only'], _ = run_one('ImageNet فقط', base_net)
            REID_RESULTS['gain_map_over_imagenet'] = REID_RESULTS['trained']['mAP'] - REID_RESULTS['imagenet_only']['mAP']
            print(f"الكسب في mAP فوق ImageNet: {REID_RESULTS['gain_map_over_imagenet']:+.4f}")
        except Exception as exc:
            status('warn', f'تعذرت مقارنة ImageNet: {type(exc).__name__}: {exc}')
    (EVAL_ROOT / 'reid_eval.json').write_text(json.dumps(REID_RESULTS, ensure_ascii=False, indent=2), encoding='utf-8')
    status('ok', 'حُفظ reid_eval.json')

✅ حُمّل reid_bot_best.pt | backbone=resnet50 | size=(224, 224) | unexpected=0
[نموذجك] mAP=0.6609 (95% CI 0.6496–0.6719) | R1=0.9124 R5=0.9559 R10=0.9690 | queries=1678
Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 128MB/s]


[ImageNet فقط] mAP=0.0989 (95% CI 0.0925–0.1053) | R1=0.3528 R5=0.5119 R10=0.5954 | queries=1678
الكسب في mAP فوق ImageNet: +0.5620
✅ حُفظ reid_eval.json


## ج) ربط المسارات بين الكاميرات (اختبار بديل على tracklets من VeRi)
- كل tracklet = مجموعة صور لمركبة واحدة في كاميرا واحدة (من `test_track.txt` إن وُجد، وإلا يُبنى من gallery بتجميع الإطارات المتقاربة).
- بصمة المسار = متوسط بصمات صوره ثم تطبيع.
- **لا نضبط العتبة على بيانات التقييم:** نقسم الهويات عشوائياً نصفين: نصف لاختيار العتبة، والنصف الآخر للقياس. نكرر ذلك عدة مرات ونعرض المتوسط والانحراف.
- حدّ هذا الاختبار: المسارات نقية الهوية بالبناء، فهو يقيس **جودة الربط البصري** فقط لا أخطاء التتبع.

In [18]:
# ١١. بناء tracklets + ضبط العتبة على dev + قياس على eval
TRACK_GAP = 150   # فجوة (بوحدات رقم الإطار في اسم VeRi) تفصل مسارين في نفس الكاميرا عند البناء الاحتياطي

def load_veri_tracklets(veri_root, gallery_df):
    names_set = set(gallery_df.name)
    track_file = next(iter(Path(veri_root).glob('test_track.txt')), None) or next(iter(Path(veri_root).rglob('test_track.txt')), None)
    tracklets, source = [], 'fallback_from_gallery'
    if track_file:
        for line in track_file.read_text(encoding='utf-8', errors='ignore').splitlines():
            names = [m.group(0) for m in VERI_RE.finditer(line)]
            names = [n for n in names if n in names_set]
            if names:
                tracklets.append(names)
        if tracklets:
            source = str(track_file)
    if not tracklets:
        for (pid, cam), grp in gallery_df.groupby(['pid', 'cam']):
            grp = grp.sort_values('ts')
            current = [grp.iloc[0]]
            for _, row in grp.iloc[1:].iterrows():
                if row.ts - current[-1].ts > TRACK_GAP:
                    tracklets.append([r['name'] for r in current]); current = []
                current.append(row)
            tracklets.append([r['name'] for r in current])
    return tracklets, source

MTMC_RESULTS = {}
if VERI is None or 'trained' not in EMB:
    status('pause', 'ربط المسارات NOT_EVALUATED (يلزم تقييم ReID أولاً)')
else:
    g = VERI[VERI.split == 'gallery'].reset_index(drop=True)
    gf = EMB['trained'][1]
    name_to_idx = {n: i for i, n in enumerate(g.name)}
    tracklets, track_source = load_veri_tracklets(VERI_ROOT, g)
    recs = []
    for tid, names in enumerate(tracklets):
        idx = [name_to_idx[n] for n in names if n in name_to_idx]
        if not idx: continue
        pids = g.pid.values[idx]; cams = g.cam.values[idx]
        vals, counts = np.unique(pids, return_counts=True)
        emb = gf[idx].mean(axis=0); emb = emb / (np.linalg.norm(emb) + 1e-12)
        recs.append({'tid': tid, 'pid': int(vals[counts.argmax()]), 'purity': float(counts.max() / len(idx)),
                     'cam': int(np.bincount(cams).argmax()), 'n': len(idx),
                     't0': int(g.ts.values[idx].min()), 't1': int(g.ts.values[idx].max()), 'emb': emb})
    T = pd.DataFrame(recs)
    print(f'tracklets={len(T)} | هويات={T.pid.nunique()} | كاميرات={T.cam.nunique()} | مصدر={track_source}')
    print(f'نقاء الهوية الوسطي داخل المسار: {T.purity.mean():.4f}')
    multi = T.groupby('pid').size()
    keep_ids = multi[multi >= 2].index                      # هوية بمسار واحد لا تُختبر على الربط
    T = T[T.pid.isin(keep_ids)].reset_index(drop=True)
    status('ok', f'بعد استبعاد الهويات ذات المسار الواحد: tracklets={len(T)} | هويات={T.pid.nunique()}')

    E = np.stack(T.emb.values)
    def eval_subset(ids, thr, camera_aware):
        sel = np.where(T.pid.isin(ids))[0]
        labels = cluster_tracklets(E[sel], T.cam.values[sel], T.t0.values[sel], T.t1.values[sel], thr, camera_aware)
        return cluster_metrics(T.pid.values[sel], labels, T.n.values[sel])

    ids_all = np.array(sorted(T.pid.unique()))
    for variant, camera_aware in (('plain', False), ('camera_aware', True)):
        runs = []
        for split_seed in range(CONFIG['mtmc_n_splits']):
            rng = np.random.default_rng(CONFIG['seed'] + split_seed)
            perm = rng.permutation(ids_all)
            dev_ids, eval_ids = perm[: len(perm) // 2], perm[len(perm) // 2:]
            scores = [(eval_subset(dev_ids, thr, camera_aware)['tracklet_idf1'], thr) for thr in CONFIG['mtmc_thr_grid']]
            best_dev_idf1, best_thr = max(scores)
            test = eval_subset(eval_ids, best_thr, camera_aware)
            test.update({'chosen_threshold': best_thr, 'dev_idf1': best_dev_idf1, 'split_seed': split_seed})
            runs.append(test)
        keys = ['tracklet_idf1', 'pairwise_f1', 'pairwise_precision', 'pairwise_recall', 'ari']
        MTMC_RESULTS[variant] = {'runs': runs,
            **{f'{k}_mean': float(np.mean([r[k] for r in runs])) for k in keys},
            **{f'{k}_std': float(np.std([r[k] for r in runs], ddof=1)) if len(runs) > 1 else 0.0 for k in keys}}
        print(f"[{variant}] IDF1={MTMC_RESULTS[variant]['tracklet_idf1_mean']:.4f}±{MTMC_RESULTS[variant]['tracklet_idf1_std']:.4f} | "
              f"pairF1={MTMC_RESULTS[variant]['pairwise_f1_mean']:.4f} | ARI={MTMC_RESULTS[variant]['ari_mean']:.4f}")
    MTMC_RESULTS['meta'] = {'tracklet_source': track_source, 'n_tracklets': int(len(T)), 'n_ids': int(T.pid.nunique()),
                            'mean_track_purity': float(np.mean([r['purity'] for r in recs])),
                            'note': 'اختبار بديل: المسارات نقية الهوية بالبناء؛ لا يشمل أخطاء التتبع ولا CityFlowV2'}
    (EVAL_ROOT / 'mtmc_proxy_eval.json').write_text(json.dumps(MTMC_RESULTS, ensure_ascii=False, indent=2, default=float), encoding='utf-8')
    status('ok', 'حُفظ mtmc_proxy_eval.json')

tracklets=2021 | هويات=200 | كاميرات=19 | مصدر=/root/.cache/kagglehub/datasets/abhyudaya12/veri-vehicle-re-identification-dataset/versions/1/VeRi/test_track.txt
نقاء الهوية الوسطي داخل المسار: 1.0000
✅ بعد استبعاد الهويات ذات المسار الواحد: tracklets=2021 | هويات=200
[plain] IDF1=0.6826±0.0278 | pairF1=0.5712 | ARI=0.5643
[camera_aware] IDF1=0.6822±0.0276 | pairF1=0.5709 | ARI=0.5639
✅ حُفظ mtmc_proxy_eval.json


## د) تتبع كاميرا واحدة (MOT) — اختياري
بيانات تتبع مركبات **بتسميات هوية** لا تُنزَّل بدون موافقة (UA-DETRAC الرسمي، KITTI Tracking، CityFlowV2). لذلك لا يخمّن الدفتر مصدراً.
ضع كل تسلسل بصيغة MOTChallenge: `seq/img1/*.jpg` و`seq/gt/gt.txt` ثم أضف المسارات إلى `MOT_SEQUENCES`.
إن بقيت القائمة فارغة تكون الحالة `NOT_EVALUATED`.

In [22]:
# ١٢. تقييم ByteTrack على تسلسلات MOTChallenge (MOTA / IDF1 / ID switches)
MOT_SEQUENCES = []    # مثال: ['/content/drive/MyDrive/YemenVision_MTMC/mot/DETRAC-MVI_20011']
MOT_RESULTS = {}

def run_mot_sequence(model_path, seq_dir, tracker='bytetrack.yaml', conf=0.25):
    import motmetrics as mm
    seq_dir = Path(seq_dir)
    images = sorted((seq_dir / 'img1').glob('*.jpg')) + sorted((seq_dir / 'img1').glob('*.png'))
    gt = pd.read_csv(seq_dir / 'gt' / 'gt.txt', header=None).iloc[:, :9]
    gt.columns = ['frame', 'id', 'x', 'y', 'w', 'h', 'flag', 'cls', 'vis']
    gt = gt[gt.flag == 1]
    acc = mm.MOTAccumulator(auto_id=False)
    model = YOLO(str(model_path))                       # نسخة جديدة لكل تسلسل = مُتتبع نظيف
    for frame_idx, result in enumerate(model.track(source=str(seq_dir / 'img1'), tracker=tracker, conf=conf,
                                                     persist=True, stream=True, verbose=False, device=DEVICE_ARG), start=1):
        g = gt[gt.frame == frame_idx]
        hyp_ids, hyp_boxes = [], []
        if result.boxes is not None and result.boxes.id is not None:
            xyxy = result.boxes.xyxy.cpu().numpy(); ids = result.boxes.id.int().cpu().numpy()
            for (x1, y1, x2, y2), tid in zip(xyxy, ids):
                hyp_ids.append(int(tid)); hyp_boxes.append([x1, y1, x2 - x1, y2 - y1])
        dists = mm.distances.iou_matrix(g[['x', 'y', 'w', 'h']].values, np.array(hyp_boxes).reshape(-1, 4), max_iou=0.5)
        acc.update(g.id.tolist(), hyp_ids, dists, frameid=frame_idx)
    summary = mm.metrics.create().compute(acc, metrics=['mota', 'idf1', 'num_switches', 'num_false_positives',
                                                         'num_misses', 'mostly_tracked', 'mostly_lost', 'precision', 'recall'], name=seq_dir.name)
    return {k: float(v) for k, v in summary.iloc[0].items()}

if MOT_SEQUENCES and DETECTOR_MODELS:
    tag = 'finetuned' if 'finetuned' in DETECTOR_MODELS else 'baseline'
    for seq in MOT_SEQUENCES:
        MOT_RESULTS[Path(seq).name] = run_mot_sequence(DETECTOR_MODELS[tag], seq)
        print(Path(seq).name, MOT_RESULTS[Path(seq).name])
    MOT_RESULTS['_mean_idf1'] = float(np.mean([v['idf1'] for k, v in MOT_RESULTS.items() if not k.startswith('_')]))
    MOT_RESULTS['_detector_used'] = tag
    (EVAL_ROOT / 'mot_eval.json').write_text(json.dumps(MOT_RESULTS, indent=2), encoding='utf-8')
else:
    status('pause', 'MOT NOT_EVALUATED — لا توجد تسلسلات MOTChallenge مُعرَّفة')

⏸️ MOT NOT_EVALUATED — لا توجد تسلسلات MOTChallenge مُعرَّفة


## هـ) التقرير النهائي وقرار المرحلة
يقرأ كل النتائج التي حُسبت فعلاً، ويقارنها بالبوابات، ويكتب `stage4_report.json` و`stage4_report.md` في Drive.

In [21]:
# ١٣. تقرير البوابات (محدث لقراءة نتائج التقييم الفعلية)
def gate(value, minimum, name, higher_is_better=True):
    if value is None:
        return {'check': name, 'status': 'NOT_EVALUATED', 'value': None, 'threshold': minimum}
    ok = value >= minimum if higher_is_better else value <= minimum
    return {'check': name, 'status': 'PASS' if ok else 'FAIL', 'value': round(float(value), 4), 'threshold': minimum}

checks = []
det_tag = 'finetuned' if 'finetuned' in DETECTOR_RESULTS else ('baseline' if 'baseline' in DETECTOR_RESULTS else None)
d = DETECTOR_RESULTS.get(det_tag) if det_tag else None
checks.append(gate(d['single_cls_all']['map50'] if d else None, GATES['detector_single_cls_map50_min'], f'detector[{det_tag}] mAP50 بلا فئة (COCO)'))
checks.append(gate(d['per_class_all']['map50'] if d else None, GATES['detector_per_class_map50_min'], f'detector[{det_tag}] mAP50 حسب الفئة (COCO)'))
checks.append(gate(d['folds_map50_std'] if d else None, GATES['detector_fold_std_max'], 'detector ثبات الأجزاء (الانحراف المعياري)', higher_is_better=False))
if 'baseline' in DETECTOR_RESULTS and 'finetuned' in DETECTOR_RESULTS:
    delta = DETECTOR_RESULTS['finetuned']['single_cls_all']['map50'] - DETECTOR_RESULTS['baseline']['single_cls_all']['map50']
    checks.append({'check': 'fine-tuned مقابل baseline (فرق mAP50 بلا فئة)', 'status': 'PASS' if delta >= 0 else 'FAIL', 'value': round(delta, 4), 'threshold': 0.0})

r = REID_RESULTS.get('trained')
checks.append(gate(r['mAP'] if r else None, GATES['reid_map_min'], 'ReID mAP (VeRi)'))
checks.append(gate(r['rank1'] if r else None, GATES['reid_rank1_min'], 'ReID Rank-1 (VeRi)'))
checks.append(gate(REID_RESULTS.get('gain_map_over_imagenet'), GATES['reid_gain_over_imagenet_map_min'], 'ReID كسب mAP فوق ImageNet'))

m = MTMC_RESULTS.get('camera_aware')
checks.append(gate(m['tracklet_idf1_mean'] if m else None, GATES['mtmc_proxy_idf1_min'], 'ربط المسارات (بديل) IDF1 على eval'))
checks.append(gate(MOT_RESULTS.get('_mean_idf1'), GATES['mot_idf1_min'], 'تتبع كاميرا واحدة IDF1'))

evaluated = [c for c in checks if c['status'] != 'NOT_EVALUATED']
overall = ('NOT_EVALUATED' if not evaluated else
           'FAIL' if any(c['status'] == 'FAIL' for c in evaluated) else
           'PASS_PARTIAL' if any(c['status'] == 'NOT_EVALUATED' for c in checks) else 'PASS')

actions = []
if d and d['single_cls_all']['map50'] < GATES['detector_single_cls_map50_min']:
    actions.append('الكاشف: أضف صوراً يمنية مُسمّاة (شوارع/إضاءة/أنواع محلية) وأعد fine-tuning؛ راجع فئات الربط في MANUAL_CLASS_MAP.')
if r and r['rank1'] < GATES['reid_rank1_min']:
    actions.append('ReID: أطل التدريب أو ارفع الإدخال إلى 256×256 أو أضف re-ranking (k-reciprocal) ثم أعد القياس على نفس query/gallery.')
if r and REID_RESULTS.get('gain_map_over_imagenet') is not None and REID_RESULTS['gain_map_over_imagenet'] < GATES['reid_gain_over_imagenet_map_min']:
    actions.append('ReID: الكسب فوق ImageNet ضعيف — تحقق من أن checkpoint هو الأفضل فعلاً ومن مطابقة التطبيع وحجم الإدخال.')
if m and m['tracklet_idf1_mean'] < GATES['mtmc_proxy_idf1_min']:
    actions.append('الربط: جرّب دمج قيود زمنية/مكانية حقيقية بين الكاميرات واختر العتبة على dev يمني.')
if not MOT_RESULTS:
    actions.append('التتبع: أضف تسلسلات MOTChallenge بتسميات هوية (UA-DETRAC/KITTI/CityFlowV2 بعد الموافقة).')
actions.append('التأكيد النهائي لليمن: ابنِ مجموعة تحقق يمنية ≥200 صورة مُسمّاة يدوياً (لم تُستخدم في التدريب) وأعد هذه الخلايا عليها.')

report = {'run_id': RUN_ID, 'overall': overall, 'checks': checks, 'next_actions': actions,
          'provenance': PROVENANCE, 'limits': [
              'COCO/VeRi ليستا بيانات يمنية.', 'ربط المسارات اختبار بديل لا MTMC رسمي.',
              'OCR اللوحات غير مشمول (لا مجموعة لوحات يمنية عامة).']}
(EVAL_ROOT / 'stage4_report.json').write_text(json.dumps(report, ensure_ascii=False, indent=2, default=str), encoding='utf-8')

lines = [f'# تقرير المرحلة ٤ — {RUN_ID}', f'**الحكم العام: {overall}**', '', '| الفحص | القيمة | الحد | الحالة |', '|---|---|---|---|']
lines += [f"| {c['check']} | {c['value']} | {c['threshold']} | {c['status']} |" for c in checks]
lines += ['', '## الإجراءات التالية'] + [f'- {a}' for a in actions]
(EVAL_ROOT / 'stage4_report.md').write_text('\n'.join(lines), encoding='utf-8')
print('\n'.join(lines))
status('ok', f'حُفظ التقرير في {EVAL_ROOT}')

# تقرير المرحلة ٤ — 20261117T191946Z
**الحكم العام: FAIL**

| الفحص | القيمة | الحد | الحالة |
|---|---|---|---|
| detector[finetuned] mAP50 بلا فئة (COCO) | 0.4926 | 0.5 | FAIL |
| detector[finetuned] mAP50 حسب الفئة (COCO) | 0.3157 | 0.4 | FAIL |
| detector ثبات الأجزاء (الانحراف المعياري) | 0.0245 | 0.03 | PASS |
| fine-tuned مقابل baseline (فرق mAP50 بلا فئة) | -0.001 | 0.0 | FAIL |
| ReID mAP (VeRi) | 0.6609 | 0.5 | PASS |
| ReID Rank-1 (VeRi) | 0.9124 | 0.8 | PASS |
| ReID كسب mAP فوق ImageNet | 0.562 | 0.15 | PASS |
| ربط المسارات (بديل) IDF1 على eval | 0.6822 | 0.6 | PASS |
| تتبع كاميرا واحدة IDF1 | None | 0.6 | NOT_EVALUATED |

## الإجراءات التالية
- الكاشف: أضف صوراً يمنية مُسمّاة (شوارع/إضاءة/أنواع محلية) وأعد fine-tuning؛ راجع فئات الربط في MANUAL_CLASS_MAP.
- التتبع: أضف تسلسلات MOTChallenge بتسميات هوية (UA-DETRAC/KITTI/CityFlowV2 بعد الموافقة).
- التأكيد النهائي لليمن: ابنِ مجموعة تحقق يمنية ≥200 صورة مُسمّاة يدوياً (لم تُستخدم في التدريب) وأعد هذه الخلايا عليها.
✅ حُفظ